# Imports and Configuration

In [ ]:

import time
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.datasets import fetch_openml
from sklearn.metrics import accuracy_score
from sklearn.neural_network import MLPClassifier

# Reproducible experimental configuration
SEEDS = [7, 21, 42]
CLASS_ORDER = list(range(20))  # MNIST 0-9, Fashion-MNIST 10-19

TRAIN_SAMPLES_PER_CLASS = 6000
EPOCHS_PER_CLASS = 15

REPLAY_BUFFER_SIZES = [100, 500, 1000, 2000]

HIDDEN_LAYER_SIZES = (256, 128)
BATCH_SIZE = 128
LEARNING_RATE = 0.001

# Save output in experiments/ when running from the project root.
OUTPUT_DIR = Path("experiments") if Path("experiments").is_dir() else Path(".")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RUNS_CSV = OUTPUT_DIR / "ann_runs.csv"
SUMMARY_TXT = OUTPUT_DIR / "ann_summary.txt"

print("Output directory:", OUTPUT_DIR.resolve())


# Data Load

In [ ]:

# Load J.A.R.V.I.S.'s existing dataset cache.
from pathlib import Path
import numpy as np

possible_paths = [
    Path("data/cache/datasets.npz"),
    Path("../data/cache/datasets.npz"),
]

CACHE_FILE = next(
    (p.resolve() for p in possible_paths if p.is_file()),
    None
)

if CACHE_FILE is None:
    raise FileNotFoundError(
        "Could not find data/cache/datasets.npz. "
        "Run this notebook from the project root or experiments folder."
    )

print("Loading cache:", CACHE_FILE)

with np.load(CACHE_FILE) as cache:
    X_train = cache["x_train"].astype(np.float32)
    y_train = cache["y_train"].astype(np.int64)
    X_test = cache["x_test"].astype(np.float32)
    y_test = cache["y_test"].astype(np.int64)

# Match DataStore's image preprocessing.
X_train = X_train.reshape(len(X_train), -1)
X_test = X_test.reshape(len(X_test), -1)

if X_train.max() > 1.0:
    X_train /= 255.0
if X_test.max() > 1.0:
    X_test /= 255.0

CLASS_ORDER = list(range(20))

train_indices = {
    c: np.flatnonzero(y_train == c)
    for c in CLASS_ORDER
}
test_indices = {
    c: np.flatnonzero(y_test == c)
    for c in CLASS_ORDER
}

print("Training shape:", X_train.shape)
print("Test shape:", X_test.shape)

print("\nAvailable examples per class:")
for c in CLASS_ORDER:
    print(
        f"Class {c:2d}: "
        f"{len(train_indices[c]):5d} train, "
        f"{len(test_indices[c]):5d} test"
    )

assert all(len(train_indices[c]) > 0 for c in CLASS_ORDER)
assert all(len(test_indices[c]) > 0 for c in CLASS_ORDER)

print("\nDataset cache loaded successfully.")


# Helper functions

In [ ]:
def select_class_examples(seed):
    """Select up to 6,000 unique training examples per class."""
    rng = np.random.default_rng(seed)
    examples = {}

    for c in CLASS_ORDER:
        idx = train_indices[c]
        n = min(TRAIN_SAMPLES_PER_CLASS, len(idx))

        selected = rng.choice(idx, size=n, replace=False)
        examples[c] = (X_train[selected], y_train[selected])

    return examples


def create_model(seed):
    """Create a conventional feed-forward ANN."""
    return MLPClassifier(
        hidden_layer_sizes=HIDDEN_LAYER_SIZES,
        activation="relu",
        solver="adam",
        learning_rate_init=LEARNING_RATE,
        batch_size=BATCH_SIZE,
        shuffle=True,
        random_state=seed,
        max_iter=1,
        tol=0.0001,  # Must be a non-negative float
        early_stopping=False
    )



def update_replay_buffer(buffer, current_x, current_y, capacity, rng):
    """
    Store actual training examples with approximately equal representation
    from every class seen so far.
    """
    if capacity <= 0:
        return {}

    current_class = int(current_y[0])
    seen_classes = sorted(set(buffer) | {current_class})

    base_quota, remainder = divmod(capacity, len(seen_classes))
    new_buffer = {}

    for position, c in enumerate(seen_classes):
        quota = base_quota + (position < remainder)

        if c == current_class:
            candidates_x, candidates_y = current_x, current_y
        else:
            candidates_x, candidates_y = buffer[c]

        take = min(int(quota), len(candidates_y))
        selected = rng.choice(len(candidates_y), size=take, replace=False)

        new_buffer[c] = (
            candidates_x[selected].copy(),
            candidates_y[selected].copy()
        )

    return new_buffer


def evaluate_model(model, seen_classes):
    """Evaluate on held-out examples for all classes introduced so far."""
    indices = np.concatenate([
        test_indices[c] for c in seen_classes
    ])

    predictions = model.predict(X_test[indices])
    overall_accuracy = accuracy_score(y_test[indices], predictions)

    per_class_accuracy = {}

    for c in seen_classes:
        idx = test_indices[c]
        per_class_accuracy[c] = accuracy_score(
            y_test[idx],
            model.predict(X_test[idx])
        )

    current_class = seen_classes[-1]
    previous_classes = seen_classes[:-1]

    previous_mean = (
        np.mean([per_class_accuracy[c] for c in previous_classes])
        if previous_classes else np.nan
    )

    return {
        "overall_accuracy": overall_accuracy,
        "new_class_accuracy": per_class_accuracy[current_class],
        "mean_previous_class_accuracy": previous_mean,
        "mean_class_accuracy": np.mean(list(per_class_accuracy.values())),
        "min_class_accuracy": min(per_class_accuracy.values()),
        "per_class_accuracy": per_class_accuracy
    }


# Run one sequential experiment

This is the main training loop. The network is initialized only once per run. It is not reset when a new class is introduced.

In [ ]:
def run_condition(seed, buffer_capacity, class_examples):
    rng = np.random.default_rng(seed + 100_000 + buffer_capacity)

    model = create_model(seed)
    replay_buffer = {}
    seen_classes = []
    results = []
    start_time = time.perf_counter()

    print(
        f"\nStarting ANN | seed={seed} | "
        f"replay buffer={buffer_capacity}"
    )

    for stage, class_id in enumerate(CLASS_ORDER, start=1):
        stage_start = time.perf_counter()

        current_x, current_y = class_examples[class_id]
        seen_classes.append(class_id)

        # Build the replay set from previously learned classes only.
        if buffer_capacity > 0 and replay_buffer:
            replay_x = np.concatenate([
                replay_buffer[c][0] for c in sorted(replay_buffer)
            ])
            replay_y = np.concatenate([
                replay_buffer[c][1] for c in sorted(replay_buffer)
            ])
        else:
            replay_x = np.empty(
                (0, current_x.shape[1]), dtype=np.float32
            )
            replay_y = np.empty((0,), dtype=np.int64)

        # Each partial_fit call performs one training epoch.
        for epoch in range(EPOCHS_PER_CLASS):
            if len(replay_y):
                epoch_x = np.concatenate([current_x, replay_x])
                epoch_y = np.concatenate([current_y, replay_y])
            else:
                epoch_x, epoch_y = current_x, current_y

            # Explicit reproducible shuffling for each epoch.
            shuffle_rng = np.random.default_rng(
                seed * 1_000_003 + stage * 10_007 + epoch
            )
            order = shuffle_rng.permutation(len(epoch_y))

            if not hasattr(model, "classes_"):
                model.partial_fit(
                    epoch_x[order],
                    epoch_y[order],
                    classes=np.asarray(CLASS_ORDER)
                )
            else:
                model.partial_fit(epoch_x[order], epoch_y[order])

        # Evaluate before updating the buffer with the current class.
        metrics = evaluate_model(model, seen_classes)
        stage_seconds = time.perf_counter() - stage_start

        row = {
            "model": "MLP_ANN",
            "replay_buffer_size": buffer_capacity,
            "replay_enabled": buffer_capacity > 0,
            "seed": seed,
            "stage": stage,
            "introduced_class": class_id,
            "classes_seen": len(seen_classes),
            "epochs_per_class": EPOCHS_PER_CLASS,
            "train_samples_per_class": TRAIN_SAMPLES_PER_CLASS,
            "overall_accuracy": metrics["overall_accuracy"],
            "new_class_accuracy": metrics["new_class_accuracy"],
            "mean_previous_class_accuracy":
                metrics["mean_previous_class_accuracy"],
            "mean_class_accuracy": metrics["mean_class_accuracy"],
            "min_class_accuracy": metrics["min_class_accuracy"],
            "stage_training_seconds": stage_seconds,
            "cumulative_elapsed_seconds":
                time.perf_counter() - start_time,
            "replay_examples_used_per_epoch": len(replay_y),
            **{
                f"class_{c}_accuracy":
                    metrics["per_class_accuracy"].get(c, np.nan)
                for c in CLASS_ORDER
            }
        }

        # Update the buffer after training and evaluation.
        if buffer_capacity > 0:
            replay_buffer = update_replay_buffer(
                replay_buffer,
                current_x,
                current_y,
                buffer_capacity,
                rng
            )

        row["buffer_examples_after_stage"] = sum(
            len(labels) for _, labels in replay_buffer.values()
        )
        results.append(row)

        print(
            f"Stage {stage:02d}/20 | class={class_id:02d} | "
            f"overall={metrics['overall_accuracy']:.4f} | "
            f"new={metrics['new_class_accuracy']:.4f} | "
            f"time={stage_seconds:.1f}s"
        )

    return results


# Complete parameter study

In [ ]:

all_results = []

# Resume safely if the notebook is interrupted: skip completed conditions.
completed_conditions = set()

if RUNS_CSV.exists():
    existing = pd.read_csv(RUNS_CSV)

    if not existing.empty:
        all_results = existing.to_dict("records")
        completed_conditions = set(
            zip(existing["seed"].astype(int),
                existing["replay_buffer_size"].astype(int))
        )
        print("Loaded", len(all_results), "existing result rows.")


for seed in SEEDS:
    # Same selected training examples across all buffer sizes for this seed.
    class_examples = select_class_examples(seed)

    for capacity in [0] + REPLAY_BUFFER_SIZES:
        condition = (seed, capacity)

        if condition in completed_conditions:
            print("Skipping completed condition:", condition)
            continue

        condition_results = run_condition(
            seed=seed,
            buffer_capacity=capacity,
            class_examples=class_examples
        )

        all_results.extend(condition_results)

        pd.DataFrame(all_results).to_csv(RUNS_CSV, index=False)
        completed_conditions.add(condition)

        print("Saved results to:", RUNS_CSV.resolve())

results_df = pd.DataFrame(all_results)

print("\nCompleted conditions:", len(completed_conditions))
print("Total stage records:", len(results_df))
display(results_df.head())


# Generate ann_summary.txt

In [ ]:

def generate_summary(df):
    lines = [
        "J.A.R.V.I.S. — Sequential ANN and Replay Study",
        "=" * 52,
        "",
        "EXPERIMENT CONFIGURATION",
        f"Class order: {CLASS_ORDER}",
        f"Seeds: {SEEDS}",
        f"Training samples per class: {TRAIN_SAMPLES_PER_CLASS}",
        f"Epochs per class: {EPOCHS_PER_CLASS}",
        f"Hidden layers: {HIDDEN_LAYER_SIZES}",
        f"Replay capacities: {REPLAY_BUFFER_SIZES}",
        "",
        "FINAL-STAGE RESULTS",
        "Mean and standard deviation across seeds",
        ""
    ]

    final_stage = df[df["stage"] == len(CLASS_ORDER)]

    for capacity in [0] + REPLAY_BUFFER_SIZES:
        subset = final_stage[
            final_stage["replay_buffer_size"] == capacity
        ]

        if subset.empty:
            continue

        label = "No replay" if capacity == 0 else f"Buffer {capacity}"

        mean_accuracy = subset["overall_accuracy"].mean()
        std_accuracy = subset["overall_accuracy"].std(ddof=1)
        if pd.isna(std_accuracy):
            std_accuracy = 0.0

        mean_new = subset["new_class_accuracy"].mean()
        mean_previous = subset["mean_previous_class_accuracy"].mean()
        mean_time = subset["cumulative_elapsed_seconds"].mean()

        lines.append(
            f"{label}: "
            f"overall={mean_accuracy:.4f} +/- {std_accuracy:.4f}; "
            f"final class={mean_new:.4f}; "
            f"previous classes={mean_previous:.4f}; "
            f"run time={mean_time:.1f}s"
        )

    lines.extend([
        "",
        "STAGE-BY-STAGE MEAN OVERALL ACCURACY",
        "Stage | Classes seen | " +
        " | ".join(
            "No replay" if c == 0 else f"Buffer {c}"
            for c in [0] + REPLAY_BUFFER_SIZES
        ),
        "-" * 85
    ])

    for stage in range(1, len(CLASS_ORDER) + 1):
        values = []

        for capacity in [0] + REPLAY_BUFFER_SIZES:
            subset = df[
                (df["stage"] == stage) &
                (df["replay_buffer_size"] == capacity)
            ]

            values.append(
                f"{subset['overall_accuracy'].mean():.4f}"
                if not subset.empty else "N/A"
            )

        lines.append(
            f"{stage:>5} | {stage:>12} | " + " | ".join(values)
        )

    lines.extend([
        "",
        "INTERPRETATION",
        "- Overall accuracy measures performance on all introduced classes.",
        "- New-class accuracy measures learning of the latest class.",
        "- Previous-class accuracy measures retention of earlier classes.",
        "- Minimum class accuracy reveals poorly retained classes.",
        "- Compare buffer sizes at matching seeds and stages.",
        "- Replay uses stored training examples; validation data is excluded.",
        "- Results are specific to this MLP architecture and training protocol."
    ])

    SUMMARY_TXT.write_text(
        "\n".join(lines) + "\n",
        encoding="utf-8"
    )

    print("Summary saved to:", SUMMARY_TXT.resolve())


generate_summary(results_df)
print("\nStudy complete.")


# Optional quick comparison

In [ ]:

comparison = (
    results_df[results_df["stage"] == len(CLASS_ORDER)]
    .groupby("replay_buffer_size")
    .agg(
        mean_accuracy=("overall_accuracy", "mean"),
        std_accuracy=("overall_accuracy", "std"),
        mean_new_class_accuracy=("new_class_accuracy", "mean"),
        mean_previous_accuracy=("mean_previous_class_accuracy", "mean"),
        mean_min_class_accuracy=("min_class_accuracy", "mean"),
        mean_runtime_seconds=("cumulative_elapsed_seconds", "mean")
    )
    .reset_index()
)

display(comparison)
